# 03. Training
Обучение PPO-агента и кривые обучения.

In [ ]:
import sys; sys.path.insert(0, "..")
import torch, torch.optim as optim, numpy as np, matplotlib.pyplot as plt, os
from rl_monomial_agent import MonomorphicOrderPolicy, RLEnvironment
from ppo_core import train_ppo
torch.manual_seed(42); np.random.seed(42)

## 1. Инициализация

In [ ]:
policy    = MonomorphicOrderPolicy(state_dim=8, num_orders=3)
optimizer = optim.Adam(policy.parameters(), lr=3e-4, weight_decay=1e-5)
env       = RLEnvironment(seed=42)
total = sum(p.numel() for p in policy.parameters())
print(f"Параметров: {total:,}")
print(f"Policy net: {sum(p.numel() for p in policy.policy_net.parameters()):,}")
print(f"Value net:  {sum(p.numel() for p in policy.value_net.parameters()):,}")

## 2. Обучение (100 эпизодов)

In [ ]:
os.makedirs("../models", exist_ok=True)
history = train_ppo(
    policy, env, optimizer,
    num_episodes=100, update_every=10,
    gamma=0.99, lam=0.95, clip_epsilon=0.2,
    value_coef=0.5, entropy_coef=0.01,
    n_epochs=4, batch_size=64,
    save_path="../models/best_model.pth",
)
torch.save(policy.state_dict(), "../models/final_model.pth")
print(f"Обновлений: {len(history)}  Финальная награда: {history[-1][chr(39)]mean_reward[chr(39)]:+.4f}")

## 3. Кривые обучения

In [ ]:
episodes = [h["episode"]     for h in history]
rewards  = [h["mean_reward"] for h in history]
p_loss   = [h["policy_loss"] for h in history]
v_loss   = [h["value_loss"]  for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, y, lbl, col in zip(axes,
    [rewards, p_loss, v_loss],
    ["Средняя награда", "Policy Loss", "Value Loss"],
    ["#2ecc71", "#e74c3c", "#3498db"]):
    ax.plot(episodes, y, "o-", color=col, lw=2, ms=6)
    ax.set_title(f"Кривая: {lbl}"); ax.set_xlabel("Эпизод"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../training_curves.png", dpi=150, bbox_inches="tight")
plt.show()